No,0.5

In [ ]:
from pathlib import Path


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

メインフォルダを選ぶと、各**末端フォルダ**（サブフォルダを1つも持たないフォルダ）に入っている
すべてのファイルを**ひとつ前（親）のフォルダ**へ移動し、空になった末端フォルダを削除します。

```
処理前                                処理後
  メイン/サブ/サブサブ/データ/a.csv  →  メイン/サブ/サブサブ/a.csv
                    （データフォルダは削除）
```

**⚠️ 実行前に必ず読んでください**

- セル **`[4]` → `[6]` の順に、同じセッションで続けて実行**してください。
  `[6]` は `[4]` が用意した対象リストを使います（02 の `[3]`→`[5]` と同じ関係）。
  カーネルを再起動した場合は `[4]` からやり直してください。
- **1回の実行で1階層だけ浅くなります。** もう一度実行すると、さらにもう1階層持ち上がります
  （元の末端フォルダが消えて、その親が新しい末端フォルダになるため）。
  意図しない限り繰り返し実行しないでください。
- 移動先に同名ファイルがある場合は、`data.csv` → `data_末端フォルダ名.csv` のように
  **末端フォルダ名を付けて**区別します。上書きは絶対にしません。
- 隠しファイル（`.DS_Store` など）は移動しません。そのため、それが残るフォルダは
  **削除されず保持**されます。
- 末端フォルダがメインフォルダの直下にある場合、ファイルはメインフォルダ直下へ移動します。

### 【1工程目】対象の末端フォルダを確認する

このセルは**ファイルを一切変更しません**。何が対象になるかを表示するだけです。
内容を確認してから、次の実行セルに進んでください。

In [13]:
# === 【1工程目】対象の末端フォルダを確認する（ファイルは変更しません） ===

main_folder_path = uu.select_folder("メインフォルダを選択してください")

terminal_folders = []

if main_folder_path is not None:

    def target_files(folder):
        """そのフォルダ直下の加工対象ファイル（隠しファイル以外）。"""
        return [p for p in sorted(folder.iterdir())
                if p.is_file() and not p.name.startswith(".")]

    # 末端フォルダ = サブフォルダを1つも持たないフォルダ
    #   uu.iter_dirs は _undo / _trash を除外して再帰的に走査する
    leaf_folders = [d for d in uu.iter_dirs(main_folder_path)
                    if not any(c.is_dir() for c in d.iterdir())]

    # 加工対象ファイルが1つ以上入っているものだけを対象にする
    terminal_folders = [d for d in leaf_folders if target_files(d)]

    if not terminal_folders:
        print("移動対象のファイルが入った末端フォルダは見つかりませんでした。")
    else:
        print(f"対象の末端フォルダ: {len(terminal_folders)} 個")
        print("=" * 70)

        total_files = 0
        renamed = 0

        # 移動先ごとに「埋まる予定の名前」を積み上げていく。
        # 同じ親フォルダを共有する兄弟の末端フォルダどうしの衝突も拾うため、
        # 末端フォルダをまたいで持ち越すのが要点（実行セルと同じ順序で処理する）。
        claimed = {}

        for terminal in terminal_folders:
            files = target_files(terminal)
            total_files += len(files)

            parent = terminal.parent
            if parent not in claimed:
                claimed[parent] = {p.name for p in parent.iterdir()}

            rel = terminal.relative_to(main_folder_path)
            parent_rel = "（メインフォルダ直下）" if parent == main_folder_path \
                else parent.relative_to(main_folder_path)

            print(f"■ {rel}")
            print(f"    対象ファイル {len(files)} 件  ->  移動先: {parent_rel}")

            # 移動先で名前が衝突するものを予告する（実行セルと同じ規則）
            for f in files:
                if f.name in claimed[parent]:
                    new_name = f"{f.stem}_{terminal.name}{f.suffix}"
                    n = 1
                    while new_name in claimed[parent]:
                        new_name = f"{f.stem}_{terminal.name}__{n}{f.suffix}"
                        n += 1
                    print(f"      ※ {f.name} は {new_name} にリネームされます")
                    claimed[parent].add(new_name)
                    renamed += 1
                else:
                    claimed[parent].add(f.name)

            # 隠しファイルが残る場合はフォルダが保持される
            hidden = [p for p in terminal.iterdir()
                      if p.is_file() and p.name.startswith(".")]
            if hidden:
                names = ", ".join(p.name for p in hidden)
                print(f"      ※ {names} が残るため、このフォルダは削除されず保持されます")

        print("=" * 70)
        print(f"合計: 末端フォルダ {len(terminal_folders)} 個 / "
              f"移動予定ファイル {total_files} 件 / リネーム {renamed} 件")
        print()
        print("内容を確認したら、次のセルを実行してください。")

選択されたフォルダ: D:/2026年度AI班/9月PP粉砕材混合材実験/加工処理データ/温度実験/NR_T
対象の末端フォルダ: 129 個
■ 0.5mm\190℃\5shot\280\2026_0916_123816_765
    対象ファイル 10 件  ->  移動先: 0.5mm\190℃\5shot\280
■ 0.5mm\190℃\5shot\320\2026_0916_124138_792
    対象ファイル 10 件  ->  移動先: 0.5mm\190℃\5shot\320
■ 0.5mm\190℃\7shot\020\2026_0916_113328_026
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\020
■ 0.5mm\190℃\7shot\040\2026_0916_120407_366
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\040
■ 0.5mm\190℃\7shot\080\2026_0916_120854_727
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\080
■ 0.5mm\190℃\7shot\120\2026_0916_121257_185
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\120
■ 0.5mm\190℃\7shot\160\2026_0916_122009_417
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\160
■ 0.5mm\190℃\7shot\200\2026_0916_122827_405
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\200
■ 0.5mm\190℃\7shot\240\2026_0916_123335_706
    対象ファイル 14 件  ->  移動先: 0.5mm\190℃\7shot\240
■ 0.5mm\210℃\5shot\280\2026_0916_141138_955
    対象ファイル 10 件  ->  移動先: 0.5mm\210℃\5shot\280
■ 0.5mm\210℃\5shot

### 【2工程目】移動と末端フォルダの削除を実行する

1工程目で表示された内容でよければ、このセルを実行します。
実行内容は記録されるので、末尾の復元セルで元に戻せます。

In [14]:
# === 【2工程目】ファイルをひとつ前のフォルダへ移動し、末端フォルダを削除 ===
# 実行内容は _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます。

STEP_NAME = "00.5_末端フォルダのファイルをひとつ前のフォルダへ移動"

if "terminal_folders" not in globals() or "main_folder_path" not in globals():
    print("エラー: 対象リストが見つかりません。先に1工程目のセルを実行してください。")
elif main_folder_path is None:
    print("エラー: メインフォルダが選択されていません。先に1工程目のセルを実行してください。")
elif not terminal_folders:
    print("対象の末端フォルダがありません。先に1工程目のセルを実行して確認してください。")
else:
    moved = 0
    deleted = 0
    kept = 0

    with uu.UndoJournal(main_folder_path, STEP_NAME) as j:
        # 1工程目で確定したリストをそのまま使う。
        # ここで末端フォルダを列挙し直すと、フォルダを消したことで親が新たな末端に
        # なってしまい、際限なく階層が潰れてしまう。
        for terminal in terminal_folders:
            if not terminal.is_dir():
                print(f"スキップ: {terminal} は既に存在しません")
                continue

            parent = terminal.parent
            files = [p for p in sorted(terminal.iterdir())
                     if p.is_file() and not p.name.startswith(".")]

            for f in files:
                dest = parent / f.name

                # 同名ファイルがある場合は末端フォルダ名を付けて区別する（上書きしない）
                if dest.exists():
                    dest = parent / f"{f.stem}_{terminal.name}{f.suffix}"
                    n = 1
                    while dest.exists():
                        dest = parent / f"{f.stem}_{terminal.name}__{n}{f.suffix}"
                        n += 1
                    print(f"  名前の衝突を回避: {f.name} -> {dest.name}")

                try:
                    j.move(f, dest)
                    moved += 1
                except Exception as e:
                    print(f"  移動エラー ({f.name}): {e}")

            # 空になっていれば削除。隠しファイル等が残っていれば保持する。
            rel = terminal.relative_to(main_folder_path)
            if j.rmdir(terminal):
                print(f"削除: {rel}")
                deleted += 1
            else:
                print(f"保持: {rel}（対象外のファイルが残っています）")
                kept += 1

    print("-" * 70)
    print(f"移動したファイル: {moved} 件")
    print(f"削除した末端フォルダ: {deleted} 個 / 保持した末端フォルダ: {kept} 個")

削除: 0.5mm\190℃\5shot\280\2026_0916_123816_765
削除: 0.5mm\190℃\5shot\320\2026_0916_124138_792
削除: 0.5mm\190℃\7shot\020\2026_0916_113328_026
削除: 0.5mm\190℃\7shot\040\2026_0916_120407_366
削除: 0.5mm\190℃\7shot\080\2026_0916_120854_727
削除: 0.5mm\190℃\7shot\120\2026_0916_121257_185
削除: 0.5mm\190℃\7shot\160\2026_0916_122009_417
削除: 0.5mm\190℃\7shot\200\2026_0916_122827_405
削除: 0.5mm\190℃\7shot\240\2026_0916_123335_706
削除: 0.5mm\210℃\5shot\280\2026_0916_141138_955
削除: 0.5mm\210℃\5shot\320\2026_0916_141648_260
削除: 0.5mm\210℃\7shot\020\2026_0916_133318_613
削除: 0.5mm\210℃\7shot\040\2026_0916_133742_349
削除: 0.5mm\210℃\7shot\080\2026_0916_134336_964
削除: 0.5mm\210℃\7shot\120\2026_0916_134916_609
削除: 0.5mm\210℃\7shot\160\2026_0916_135337_949
削除: 0.5mm\210℃\7shot\200\2026_0916_135827_564
削除: 0.5mm\210℃\7shot\240\2026_0916_140410_975
削除: 0.5mm\230℃\5shot\280\2026_0916_155540_973
削除: 0.5mm\230℃\5shot\320\2026_0916_160030_071
削除: 0.5mm\230℃\7shot\020\2026_0916_151136_546
削除: 0.5mm\230℃\7shot\040\2026_0916

---
### 🔎 現在のフォルダ構成を調べる（確認用・いつでも実行できます）

このセルは**ファイルを一切変更しません**。次の3つを表示します。

1. 現在のフォルダ構成（メインフォルダからの深さごとの集計）
2. 「02_ファイル分別」 を実行したらどうなるか**の予測（実際には動かさずシミュレートします）
3. その後 04・05 が付けるファイル名の予測

工程の順序を入れ替えたとき（例: 01 を飛ばしてこのノートブックを先に実行したとき）に、
02 のセル `[7]` や 04 の命名が意図どおりになるかを、実行前に確かめるためのものです。

In [ ]:
# === 🔎 現在のフォルダ構成を調べる（ファイルは変更しません） ===
# 02 を実行した場合の結果をシミュレートして表示します。実際には何も動かしません。

from collections import defaultdict

SAMPLE = 3  # 各項目で表示するサンプル件数


def normalize_for_step7(rel_paths):
    """02 のセル [7]（拡張子フォルダの集約）と同じ規則で配置を計算する。"""
    out = []
    for p in rel_paths:
        ext_name = p.suffix[1:]
        parts = [q for q in p.parent.parts if q != ext_name]
        out.append(Path(ext_name).joinpath(*parts, p.name))
    return out


def diagnose():
    root = uu.select_folder("調べたいメインフォルダを選択してください")
    if root is None:
        return None

    # ---------------------------------------------------------------
    # 現在の状態を「メインフォルダからの相対パス」の集合として取り込む
    # ---------------------------------------------------------------
    files = []   # 可視ファイル（_undo / _trash と隠しファイルは除外）
    dirs = []
    for p in sorted(root.rglob("*")):
        if uu.is_reserved(p, root):
            continue
        rel = p.relative_to(root)
        if p.is_dir():
            dirs.append(rel)
        elif not p.name.startswith("."):
            files.append(rel)

    if not files:
        print("可視ファイルが1件も見つかりませんでした。")
        return None

    # ---------------------------------------------------------------
    # 1. 現在の構成を深さ別に集計
    # ---------------------------------------------------------------
    print("=" * 72)
    print("【1】現在のフォルダ構成")
    print("=" * 72)
    print(f"メインフォルダ: {root}")
    print()

    dirs_by_depth = defaultdict(list)
    for d in dirs:
        dirs_by_depth[len(d.parts)].append(d)

    files_by_depth = defaultdict(list)
    for f in files:
        files_by_depth[len(f.parts) - 1].append(f)

    for depth in sorted(set(dirs_by_depth) | set(files_by_depth)):
        nd = len(dirs_by_depth.get(depth, []))
        nf = len(files_by_depth.get(depth, []))
        label = "メインフォルダ直下" if depth == 0 else f"深さ {depth}"
        print(f"  {label}: フォルダ {nd} 個 / ファイル {nf} 件")
        for d in dirs_by_depth.get(depth, [])[:SAMPLE]:
            print(f"      └ {d}")

    print()
    depths = sorted(files_by_depth)
    if len(depths) == 1:
        d = depths[0]
        print(f"  ファイルはすべて深さ {d} にあります"
              f"（メイン/{'/'.join(['□'] * d)}/*.csv）")
    else:
        print(f"  ⚠ ファイルの深さが揃っていません: {depths}")
        print("     04 の命名は深さを前提にしているため、")
        print("     このまま進めると一部だけ意図しない結果になる可能性があります。")

    # ---------------------------------------------------------------
    # 2. 02 を実行した場合をシミュレート（ディスクには触れない）
    # ---------------------------------------------------------------
    print()
    print("=" * 72)
    print("【2】02 を実行した場合の予測（シミュレーション）")
    print("=" * 72)

    # --- セル [3]: 拡張子ごとの仕分け ---
    #     メイン/<拡張子>/<現在の親の相対パス>/<ファイル名>
    after3 = []
    for f in files:
        ext = f.suffix
        if not ext:
            continue  # 拡張子が無いファイルは触らない
        parent = f.parent
        base = Path(ext[1:])
        after3.append(base / f.name if str(parent) == "."
                      else base / parent / f.name)

    print(f"  [3] 仕分け        : {len(after3)} 件のファイルを移動")
    for p in after3[:SAMPLE]:
        print(f"      └ {p}")

    # --- セル [5]: 可視ファイルを含まなくなった直下フォルダを削除 ---
    top_with_files = {p.parts[0] for p in after3}
    removed_tops = sorted({d.parts[0] for d in dirs} - top_with_files)
    print(f"  [5] 空フォルダ削除: 直下の {len(removed_tops)} 個のフォルダツリーを削除")
    for t in removed_tops[:SAMPLE]:
        print(f"      └ {t}")

    # --- セル [7]: 拡張子フォルダをメインフォルダ直下に集約 ---
    final = normalize_for_step7(after3)
    n_moved = sum(1 for a, b in zip(after3, final) if a != b)
    print(f"  [7] 最終段階      : {n_moved} 件のファイルを移動"
          "（拡張子フォルダをメイン直下へ集約）")
    if n_moved == 0:
        print("      ※ [3] の時点で既に正しい配置のため 0 件です（正常です）")
    else:
        for a, b in list(zip(after3, final))[:SAMPLE]:
            if a != b:
                print(f"      └ {a}  ->  {b}")

    print()
    print("  02 をすべて実行したあとのファイル配置（予測）:")
    for p in final[:SAMPLE]:
        print(f"      └ {p}")
    fdepths = sorted({len(p.parts) - 1 for p in final})
    print(f"  ファイルの深さ: {fdepths}")

    # ---------------------------------------------------------------
    # 3. 04・05 が付けるファイル名の予測
    # ---------------------------------------------------------------
    print()
    print("=" * 72)
    print("【3】そのあと 04・05 が付けるファイル名の予測")
    print("=" * 72)

    result = {"root": root, "files_now": files,
              "after_02": final, "step7_moves": n_moved}

    sample = next((p for p in final if len(p.parts) >= 3), None)
    if sample is None:
        print("  ⚠ 階層が浅すぎて、04 の命名規則（親・祖父フォルダ名を使う）が")
        print("     成り立ちません。")
        return result

    terminal = sample.parent.name              # 末端フォルダ名
    pre_terminal = sample.parent.parent.name   # そのひとつ前のフォルダ名
    print(f"  例にするファイル: {sample}")
    print(f"    末端フォルダ名          = {terminal}")
    print(f"    そのひとつ前のフォルダ名 = {pre_terminal}")
    print()
    print(f"  04 が付ける名前: 0<連番>_{pre_terminal}_{terminal}.csv")
    print(f"  05 が付ける名前: <元の名前>_{pre_terminal}.csv")
    print()
    if "csv" in (terminal, pre_terminal):
        print("  ⚠ 拡張子フォルダ 'csv' がファイル名に混ざります。")
        print("     意図した命名にならない可能性が高いです。")
    else:
        print("  上の2つの名前が意図どおりであれば、このまま進めて問題ありません。")

    return result


# 結果は diagnosis_result に入るので、あとから中身を確認できます
diagnosis_result = diagnose()

---
### ⏪ 復元（元に戻す）


このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）


繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()

---
### 🧹 `_undo` と `_trash` を片付ける（すべての工程が終わってから）

メインフォルダの `_undo`（復元用の履歴）と `_trash`（削除の代わりの退避先）を削除します。

**処理の途中では実行しないでください。** 削除すると、それまでの工程を**元に戻せなくなります**。
`_trash` に退避中のファイルがある場合、それらは完全に失われます。

安全のため、既定では**削除せずに中身を報告するだけ**です。
内容を確認したうえで、セル先頭の `CONFIRM` を `True` に変えて実行してください。

In [ ]:
# === 🧹 _undo と _trash を片付ける（すべての工程が終わってから） ===

# ==========================================
# 設定
# ==========================================
CONFIRM = False   # False = 報告のみ（削除しません） / True = 実際に削除します
# ==========================================

import json
import shutil


def cleanup():
    root = uu.select_folder("片付けたいメインフォルダを選択してください")
    if root is None:
        return

    undo_dir = root / uu.UNDO_DIR_NAME
    trash_dir = root / uu.TRASH_DIR_NAME

    print("=" * 70)
    print("現在の状態")
    print("=" * 70)

    # --- _undo の中身 ---
    log_path = undo_dir / uu.LOG_FILE_NAME
    if log_path.exists():
        steps = json.load(open(log_path, encoding="utf-8"))["steps"]
        print(f"  _undo : 履歴 {len(steps)} ステップ")
        for i, s in enumerate(steps, start=1):
            print(f"            {i:>2}. {s['step']}（{s['timestamp']}）")
    elif undo_dir.exists():
        print("  _undo : フォルダはありますが履歴はありません")
    else:
        print("  _undo : ありません")

    # --- _trash の中身 ---
    trash_files = [p for p in trash_dir.rglob("*") if p.is_file()] \
        if trash_dir.is_dir() else []
    total = sum(p.stat().st_size for p in trash_files)
    if trash_files:
        print(f"  _trash: ファイル {len(trash_files)} 件 / 合計 {total / 1024:,.1f} KB")
        for p in trash_files[:5]:
            print(f"            {p.relative_to(trash_dir)}")
        if len(trash_files) > 5:
            print(f"            ... ほか {len(trash_files) - 5} 件")
    elif trash_dir.exists():
        print("  _trash: 空です")
    else:
        print("  _trash: ありません")

    if not undo_dir.exists() and not trash_dir.exists():
        print()
        print("片付けるものはありません。")
        return

    print()
    if not CONFIRM:
        print("-" * 70)
        print("CONFIRM = False のため、まだ何も削除していません。")
        if trash_files:
            print(f"⚠ _trash に {len(trash_files)} 件のファイルが退避されています。")
            print("   削除すると、これらは完全に失われます。")
            print("   必要なファイルが無いか、先に中身を確認してください。")
        if log_path.exists():
            print("⚠ _undo を削除すると、これまでの工程を元に戻せなくなります。")
        print()
        print("問題なければ、このセルの先頭を CONFIRM = True に変えて実行してください。")
        return

    # --- 実際に削除する ---
    print("-" * 70)
    print("削除を実行します...")
    for d in (undo_dir, trash_dir):
        if d.exists():
            shutil.rmtree(d)
            print(f"  削除しました: {d.name}")

    print()
    print("片付けが完了しました。")
    print("これ以降、このフォルダでは復元セルを使えません。")


cleanup()